<a href="https://colab.research.google.com/github/vicky-27930/My-Projects/blob/project-branch/Fake_News_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression  # New Import
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
print("Loading datasets...")
# Using the specific file names provided
df_true = pd.read_csv('true.csv')
df_fake = pd.read_csv('fake.csv')
df_eval = pd.read_csv('transformed_responses.csv')

Loading datasets...


In [ ]:
print("Preparing training data...")
df_true['label'] = 1
df_fake['label'] = 0

df_train = pd.concat([df_true, df_fake], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
df_train['content'] = df_train['title'].astype(str) + " " + df_train['text'].astype(str)
df_train = df_train[['content', 'label']]

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\n', ' ', text)
    return text

print("Cleaning text data...")
df_train['content_cleaned'] = df_train['content'].apply(clean_text)
df_eval['statement_cleaned'] = df_eval['statement'].apply(clean_text)

Preparing training data...
Cleaning text data...


In [ ]:
print("Vectorizing text...")
X_train, X_test, y_train, y_test = train_test_split(
    df_train['content_cleaned'], df_train['label'], test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer(stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)
X_eval_tfidf = vectorizer.transform(df_eval['statement_cleaned'])


Vectorizing text...


In [ ]:
# SVM: Keep at standard strength
print("\n--- Training Support Vector Machine (LinearSVC) ---")
svm_model = LinearSVC(C=1.0, max_iter=1000, dual='auto', random_state=42)
svm_model.fit(X_train_tfidf, y_train)

# Logistic Regression: Apply HEAVY regularization (C=0.01)
# This forces the model to be simpler and likely change its edge-case predictions
print("--- Training Logistic Regression ---")
lr_model = LogisticRegression(C=0.01, max_iter=1000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)


--- Training Support Vector Machine (LinearSVC) ---
--- Training Logistic Regression ---


LogisticRegression(C=0.01, max_iter=1000, random_state=42)

In [ ]:
print("\n--- Predicting on Real-Time Form Responses ---")
df_eval['svm_prediction_num'] = svm_model.predict(X_eval_tfidf)
df_eval['lr_prediction_num'] = lr_model.predict(X_eval_tfidf)

# Map labels for display
label_mapping_text = {1: 'Real', 0: 'Fake'}
df_eval['SVM_Prediction'] = df_eval['svm_prediction_num'].map(label_mapping_text)
df_eval['LR_Prediction'] = df_eval['lr_prediction_num'].map(label_mapping_text)

# Display comparisons
df_eval_display = df_eval[['statement', 'human_consensus', 'SVM_Prediction', 'LR_Prediction']]
print("\nFinal Results on the Real-time Dataset:\n")
print(df_eval_display.to_string())


--- Predicting on Real-Time Form Responses ---

Final Results on the Real-time Dataset:

                                                                                                    statement human_consensus SVM_Prediction LR_Prediction
0                            Scientists announced they developed a pill that can cure all diseases instantly.            Fake           Fake          Fake
1                           India successfully launched a communication satellite using ISRO’s latest rocket.            Real           Real          Fake
2                                             A secret underground city was discovered beneath the Taj Mahal.            Fake           Fake          Fake
3                                    Stock markets showed significant growth after positive economic reports.            Real           Real          Real
4   A study published on February 2, 2025 claims drinking hot water every hour prevents all viral infections.            Fake          

In [ ]:
eval_label_mapping = {'Real': 1, 'Fake': 0}
df_eval['human_consensus_num'] = df_eval['human_consensus'].map(eval_label_mapping)
y_true_eval = df_eval['human_consensus_num']

def print_metrics(y_true, y_pred, model_name):
    print(f"\n--- {model_name} Evaluation (vs. Human Consensus) ---")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred) * 100:.2f}%")
    print(f"Precision: {precision_score(y_true, y_pred) * 100:.2f}%")
    print(f"Recall:    {recall_score(y_true, y_pred) * 100:.2f}%")
    print(f"F1-Score:  {f1_score(y_true, y_pred) * 100:.2f}%")

print_metrics(y_true_eval, df_eval['svm_prediction_num'], "SVM")
print_metrics(y_true_eval, df_eval['lr_prediction_num'], "Logistic Regression")


--- SVM Evaluation (vs. Human Consensus) ---
Accuracy:  90.00%
Precision: 81.82%
Recall:    100.00%
F1-Score:  90.00%

--- Logistic Regression Evaluation (vs. Human Consensus) ---
Accuracy:  80.00%
Precision: 100.00%
Recall:    55.56%
F1-Score:  71.43%
